In [1]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql.types import LongType, StringType, StructField, StructType, BooleanType, ArrayType, IntegerType, DoubleType

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration for the EDSTD project") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
spark.sql(
    """
    SHOW TABLES FROM projeto
    """
).show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|  projeto|     amazon_books|      false|
|  projeto|   amazon_credits|      false|
|  projeto|    amazon_titles|      false|
|  projeto| book_movie_adapt|      false|
|  projeto|  netflix_credits|      false|
|  projeto|   netflix_titles|      false|
|  projeto|streaming_credits|      false|
|  projeto| streaming_titles|      false|
+---------+-----------------+-----------+



In [16]:
# spark.sql(
#    """
#    DROP TABLE IF EXISTS projeto.netflix_titles
#    """
#)

DataFrame[]

In [ ]:
spark.sql("""
    CREATE EXTERNAL TABLE projeto.netflix_titles (
        id VARCHAR(50),
        title VARCHAR(130),
        type VARCHAR(5),
        description VARCHAR(2200),
        release_year INT,
        age_certification VARCHAR(5),
        runtime INT,
        production_countries VARCHAR(50),
        seasons DOUBLE,
        imdb_id VARCHAR(10),
        imdb_score DOUBLE,
        imdb_votes DOUBLE,
        tmdb_popularity DOUBLE,
        tmdb_score DOUBLE,
        genre_scifi integer,
        genre_documentation integer,
        genre_crime integer,
        genre_fantasy integer,
        genre_action integer,
        genre_animation integer,
        genre_sport integer, 
        genre_family integer, 
        genre_horror integer,
        genre_history integer, 
        genre_music integer,
        genre_drama integer, 
        genre_western integer, 
        genre_war integer, 
        genre_european integer, 
        genre_romance integer, 
        genre_thriller integer, 
        genre_reality integer, 
        genre_comedy integer 
    )
    USING DELTA
    PARTITIONED BY (release_year)
    LOCATION 'hdfs://hdfs-nn:9000/warehouse/projeto.db/netflix_titles'
"""
)

In [5]:
spark.sql(
    """
    DESCRIBE FORMATTED projeto.netflix_titles
    """
).toPandas()

,col_name,data_type,comment
0,id,string,None
1,title,string,None
2,type,string,None
3,description,string,None
4,release_year,int,None
5,age_certification,string,None
6,runtime,int,None
7,seasons,int,None
8,imdb_id,string,None
9,imdb_score,double,None


In [6]:
spark.sql(
    """
    SELECT * FROM projeto.netflix_titles
    """
).show()

+--------+--------------------+-----+--------------------+------------+-----------------+-------+-------+---------+----------+----------+---------------+----------+-----------+-------------------+-----------+-------------+------------+---------------+-----------+------------+------------+-------------+-----------+-----------+-------------+---------+--------------+-------------+--------------+-------------+------------+--------------------+
|      id|               title| type|         description|release_year|age_certification|runtime|seasons|  imdb_id|imdb_score|imdb_votes|tmdb_popularity|tmdb_score|genre_scifi|genre_documentation|genre_crime|genre_fantasy|genre_action|genre_animation|genre_sport|genre_family|genre_horror|genre_history|genre_music|genre_drama|genre_western|genre_war|genre_european|genre_romance|genre_thriller|genre_reality|genre_comedy|production_countries|
+--------+--------------------+-----+--------------------+------------+-----------------+-------+-------+-------

In [22]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.netflix_titles_v2
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-11-15 16:34:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 63, ...|        null|Apache-Spark/3.4....|
|      0|2025-11-15 16:33:...|  null|    null|        CREATE TABLE|{isManaged -> fal...|null|    null|     null|       null|  Serializable|         true|           

In [ ]:
spark.stop()